In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

In [18]:
@dataclass
class GQAConfig:
    hidden_dim: int
    q_heads: int
    kv_heads: int
    dropout: float


class GroupQueryAttention(nn.Module):
    def __init__(self, config: GQAConfig):
        super().__init__()
        self.config = config
        self.q_heads = config.q_heads
        self.kv_heads = config.kv_heads
        self.head_dim = config.hidden_dim // config.q_heads
        self.q_proj = nn.Linear(config.hidden_dim, config.q_heads * self.head_dim)
        self.k_proj = nn.Linear(config.hidden_dim, config.kv_heads * self.head_dim)
        self.v_proj = nn.Linear(config.hidden_dim, config.kv_heads * self.head_dim)
        self.o_proj = nn.Linear(config.hidden_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask=None):
        # x shape (batch, seq_len, hidden_dim)
        batch, seq_len, _ = x.size()
        q = self.q_proj(x)  # (batch, seq_len, q_heads * head_dim)
        k = self.k_proj(x)  # (batch, kv_heads * head_dim)
        v = self.v_proj(x)  # # (batch, kv_heads * head_dim)

        q = q.view(batch, seq_len, self.q_heads, self.head_dim).transpose(1, 2)   #(batch, q_heads, seq_len, head_dim)
        k = k.view(batch, seq_len, self.kv_heads, self.head_dim).transpose(1, 2)  #(batch, kv_heads, seq_len, head_dim)
        v = v.view(batch, seq_len, self.kv_heads, self.head_dim).transpose(1, 2)  #(batch, kv_heads, seq_len, head_dim)

        k = k.repeat_interleave(self.q_heads // self.kv_heads, dim=1)  
        v = v.repeat_interleave(self.q_heads // self.kv_heads, dim=1) 

        attention_weight = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)  # (batch, q_heads, seq_len, seq_len)
        if attention_mask is not None:
            attention_weight.masked_fill(
                attention_mask == 0,
                float('-inf')
            )
        attention_weight = F.softmax(attention_weight, dim=-1)
        attention_weight = self.dropout(attention_weight)
        output = attention_weight @ v  #(batch, q_heads, seq_len, head_dim)
        output = output.transpose(1, 2).contiguous()  #(batch, seq_len, q_heads, head_dim)
        output = output.view(batch, seq_len, -1)  #(batch, seq_len, hidden_dim)
        return self.o_proj(output)

        
        


In [19]:
x = torch.rand(2, 4, 24)
config = GQAConfig(hidden_dim=24, q_heads=4, kv_heads=2, dropout=0.1)
GQA = GroupQueryAttention(config)
attention_mask = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0]
]).unsqueeze(1).unsqueeze(2).expand(2, 4, 4, 4)
attention_mask.shape
output = GQA(x, attention_mask=attention_mask)
output

tensor([[[ 6.2322e-03,  9.7591e-02,  1.2324e-01,  9.7679e-02, -4.5403e-02,
          -7.8357e-02,  3.9417e-01, -3.0856e-02,  2.0782e-01, -1.3584e-01,
          -1.5659e-01,  1.7139e-01,  1.1692e-01, -6.4545e-02,  1.1062e-01,
           2.0068e-01, -1.9651e-01,  1.4926e-01,  8.2003e-02,  1.6390e-02,
          -1.8442e-01,  2.2221e-01,  2.1089e-01, -3.2394e-01],
         [ 4.5167e-02, -1.1865e-02,  6.8434e-02,  9.5777e-02, -7.1733e-02,
          -1.4213e-01,  4.6176e-01, -5.8525e-02,  2.0163e-01, -1.0657e-01,
          -6.5478e-02,  6.0663e-02,  1.1098e-01, -6.0101e-02,  2.1122e-01,
           1.5085e-01, -1.4203e-01,  1.3828e-01,  1.5172e-02,  2.5212e-02,
          -1.4296e-01,  1.3916e-01,  1.8912e-01, -3.1561e-01],
         [ 9.4928e-03,  6.0022e-02,  1.4874e-01,  9.9493e-02,  1.1902e-02,
          -3.4652e-02,  3.9273e-01, -5.9490e-02,  2.6445e-01, -1.1870e-01,
          -1.1706e-01,  4.5458e-02,  1.9297e-01, -1.5335e-02,  2.3536e-01,
           1.7709e-01, -1.3063e-01,  4.3420e-02, 

In [2]:
def reciprocal_rank_fusion(rankings, k=60):
    """
    对多个排序列表进行 RRF 融合。

    Args:
        rankings: list of list, 每个子列表按相关性降序排列的 doc_id (排名1最相关)
        k: 平滑常数，默认 60

    Returns:
        list of tuple: [(doc_id, score), ...] 按融合得分降序排列
    """
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# 简单的面试测试
if __name__ == "__main__":
    # 模拟两个检索系统对同批文档的排名
    sys1 = ["A", "B", "C", "D"]   # A 最相关
    sys2 = ['C', "B", "A", 'D']   # B 最相关

    fused = reciprocal_rank_fusion([sys1, sys2], k=60)
    for doc, score in fused:
        print(f"Doc {doc}: {score:.6f}")

Doc A: 0.032266
Doc C: 0.032266
Doc B: 0.032258
Doc D: 0.031250
